# CAB: live timing on a GPU, part 2 (PROTOCOL.md, protocol v3, addendum C)

Times SC2-PCR and TurboReg (full budget and the two certified CAB configurations) on the six remaining test sets of addendum C: KITTI raw drives, 3DMatch, 3DLoMatch, KITTI odometry, FCGF 3DMatch and FCGF 3DLoMatch, on the same 300 test pairs per set as the CPU live timing. Nothing is trained here.

**Steps**
1. Upload `gpu_bundle_part2.zip` to the top level of your Google Drive ("My Drive").
2. Choose a GPU runtime: in Colab, *Runtime > Change runtime type > T4 GPU*; in VS Code with the Colab extension, select a Colab **GPU** server as the kernel.
3. *Run all*. The second cell asks for the GitHub token right away and checks it (a fine-grained token restricted to this repository with *Contents: Read and write*; or store it as the Colab secret `GITHUB_TOKEN`). Then everything runs unattended, about 1.5-2.5 hours on a T4.
4. At the end the results are copied to Drive and published to the GitHub repository (`results/gpu/`). Never put the token in this notebook's code.


In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: select a GPU runtime first (see step 2)"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
# GitHub token, asked once at the start and checked now, used at the end of each phase to publish the results
import getpass, json, urllib.request
REPO = "AFAcoding/Certified-Adaptive-Budgets-for-Robust-Point-Cloud-Registration"
try:
    from google.colab import userdata
    _token = userdata.get("GITHUB_TOKEN")
except Exception:
    _token = None
if not _token:
    _token = getpass.getpass("GitHub token (fine-grained, this repository, Contents: read and write): ").strip()
_req = urllib.request.Request(f"https://api.github.com/repos/{REPO}",
                              headers={"Authorization": f"Bearer {_token}", "Accept": "application/vnd.github+json"})
with urllib.request.urlopen(_req) as _r:
    print("token OK for", json.load(_r)["full_name"], "- you can leave the notebook running")

In [ ]:
%pip install -q igraph

In [ ]:
import os, zipfile
ZIP = "/content/drive/MyDrive/gpu_bundle_part2.zip"
if not os.path.exists("/content/gpu_bundle_part2/config.json"):
    if not os.path.exists(ZIP):
        # VS Code: if this fails, run the command "Colab: Mount Google Drive to Server", then run this cell again
        from google.colab import drive
        drive.mount("/content/drive")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/content")
print(sorted(os.listdir("/content/gpu_bundle_part2")))

In [ ]:
import base64, gzip, shutil, time

def publish(path):
    """Copy the results to Drive and publish them (gzipped) to the GitHub repository, results/gpu/."""
    if os.path.isdir("/content/drive/MyDrive"):
        shutil.copy(path, "/content/drive/MyDrive/")
        print("copied to Drive:", os.path.basename(path))
    name = f"results/gpu/{os.path.basename(path)[:-5]}_{time.strftime('%Y%m%d_%H%M%S')}.json.gz"
    body = json.dumps({"message": f"GPU timing results: {os.path.basename(name)}",
                       "content": base64.b64encode(gzip.compress(open(path, "rb").read())).decode()}).encode()
    req = urllib.request.Request(f"https://api.github.com/repos/{REPO}/contents/{name}", data=body, method="PUT",
                                 headers={"Authorization": f"Bearer {_token}", "Accept": "application/vnd.github+json",
                                          "X-GitHub-Api-Version": "2022-11-28"})
    with urllib.request.urlopen(req) as r:
        print("published:", json.load(r)["content"]["html_url"])

## SC2-PCR and TurboReg on the six test sets of part 2


In [ ]:
!cd /content/gpu_bundle_part2 && python -u run_gpu_timing.py --bundle . --estimators sc2,turbo --out /content/cab_gpu_results_sc2_turbo_part2.json
publish("/content/cab_gpu_results_sc2_turbo_part2.json")
